In [ ]:
%pip install -e ..

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from simple_transformer.checkpoint import CheckpointConfig, CheckpointManager
from simple_transformer.config import local_training_config, small_model_config
from simple_transformer.metrics import TensorBoardTrainingObserver
from simple_transformer.model import SimpleTransformerLM, count_parameters
from simple_transformer.data import make_train_val_loaders
from simple_transformer.train import fit


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.__version__)
print(device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
custom_examples = [
    "0+0=",
    "5+7=",
    "9+1=",
    "10+5=",
    "45+54=",
    "99+1=",
    "123+456=",
    "999+1=",
    "745+84=",
    "12+385=",
    "1000+0=",
    "1005+2000=",
    "1234+1111=",
    "4096+2048=",
    "9999+1=",
    "9999+9999=",
]

for example in custom_examples:
    print(example)

In [ ]:
def show_generations(model, tokenizer, examples):
    model.eval()
    prompt_ids = [tokenizer.encode(example) for example in examples]
    generated_ids = model.generate_batch(
        prompt_ids,
        eos_token_id=tokenizer.eos_token_id,
    )

    for prompt, output_ids in zip(examples, generated_ids):
        print(f"{prompt:>8} -> {tokenizer.decode(output_ids)}")

In [ ]:
train_config = local_training_config(max_digits=4, device=device)

model_config = small_model_config(
    max_digits=train_config.max_digits,
    device=train_config.device,
)
model = SimpleTransformerLM(model_config)
train_loader, val_loader, tokenizer = make_train_val_loaders(train_config)
run_name = datetime.now().strftime("addition-%Y%m%d-%H%M%S")
run_dir = Path("..") / "runs" / run_name
checkpoint_dir = Path("..") / "checkpoints" / run_name
checkpoint_manager = CheckpointManager(
    CheckpointConfig(checkpoint_dir=checkpoint_dir, keep_last=3)
)
observer = TensorBoardTrainingObserver(run_dir)
observer.log_config(
    training_config=train_config,
    model_config=model_config,
    parameter_count=count_parameters(model),
)

print(f"parameters: {count_parameters(model):,}")
print(f"force flash: {model_config.force_flash}")
print(f"train batches: {len(train_loader)}, validation batches: {len(val_loader)}")
print(f"TensorBoard log dir: {run_dir}")
print(f"Checkpoint dir: {checkpoint_dir}")
print("Terminal: tensorboard --logdir runs")
print("Before training:")
show_generations(model, tokenizer, custom_examples)


In [ ]:
def print_epoch(epoch, train, validation):
    print(
        f"epoch {epoch:02d} | "
        f"train loss {train.loss:.4f} acc {train.accuracy:.3f} | "
        f"val loss {validation.loss:.4f} acc {validation.accuracy:.3f} | "
        f"lr {train.learning_rate:.2e}"
    )

try:
    history = fit(
        model,
        train_loader,
        val_loader,
        train_config,
        on_epoch=print_epoch,
        observer=observer,
        checkpoint_manager=checkpoint_manager,
    )
finally:
    observer.close()
    checkpoint_manager.close()


To resume this same run after an interruption, recreate the model, loaders, observer, and checkpoint manager, then pass the latest checkpoint back into `fit`:

```python
latest_checkpoint = checkpoint_manager.latest_checkpoint()
history = fit(
    model,
    train_loader,
    val_loader,
    train_config,
    on_epoch=print_epoch,
    observer=observer,
    checkpoint_manager=checkpoint_manager,
    resume_from=latest_checkpoint,
)
```


In [ ]:
epochs = range(1, len(history.train) + 1)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, [metrics.loss for metrics in history.train], label="train")
plt.plot(epochs, [metrics.loss for metrics in history.validation], label="validation")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, [metrics.accuracy for metrics in history.train], label="train")
plt.plot(epochs, [metrics.accuracy for metrics in history.validation], label="validation")
plt.xlabel("epoch")
plt.ylabel("token accuracy")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()

In [ ]:
print("After training:")
show_generations(model, tokenizer, custom_examples)

final_train = history.train[-1]
final_validation = history.validation[-1]
print(
    f"final train loss={final_train.loss:.4f}, "
    f"train acc={final_train.accuracy:.3f}, "
    f"val loss={final_validation.loss:.4f}, "
    f"val acc={final_validation.accuracy:.3f}"
)

In [ ]:
run_name = "addition-20260619-183846"
run_dir = Path("..") / "runs" / run_name
checkpoint_dir = Path("..") / "checkpoints" / run_name
checkpoint_manager = CheckpointManager(
    CheckpointConfig(checkpoint_dir=checkpoint_dir, keep_last=3)
)
latest_checkpoint = checkpoint_manager.latest_checkpoint()
print(f"latest checkpoint: {latest_checkpoint}\n")

inference_model = SimpleTransformerLM(model_config)
checkpoint_manager.load_checkpoint(
    latest_checkpoint,
    model=inference_model,
    map_location=device,
)
inference_model = inference_model.to(device)

show_generations(inference_model, tokenizer, custom_examples)